# Capítulo 10 — Arquitetura de aplicações de IA e feedback do usuário

**Livro:** começa na página **807** do PDF.

📗 **Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Passo 2. Instale os guarda-corpos. · PDF p. 811

#### 🧪 Exercício: Guarda-corpo: esconder dados pessoais

Antes de mandar um texto ao modelo, mascare **e-mails e CPFs** com expressões regulares (`re.sub`). Teste com uma mensagem que tenha os dois.

In [ ]:
import re
def mascarar(texto):
    texto = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", texto)
    texto = re.sub(r"\d{3}\.?\d{3}\.?\d{3}-?\d{2}", "[CPF]", texto)
    return texto

msg = "Sou o João, CPF 123.456.789-00, e-mail joao.silva@empresa.com. Meu pedido não chegou."
seguro = mascarar(msg)
print(seguro)
print(perguntar(f"Resuma o problema do cliente em uma frase: {seguro}"))

### Etapa 3. Adicione o roteador e o gateway do modelo. · PDF p. 819

#### 🧪 Exercício: Roteador: modelo barato ou forte?

Use o modelo barato para classificar cada pergunta como **simples** ou **complexa**. Mande as simples para `MODELO` e as complexas para `MODELO_FORTE`. Teste com 4 perguntas.

In [ ]:
def rotear(pergunta):
    tipo = perguntar(f"A pergunta é simples (fato curto) ou complexa (exige raciocínio ou análise)? "
                     f"Responda só 'simples' ou 'complexa'.\n\n{pergunta}", temperature=0).strip().lower()
    modelo = MODELO_FORTE if "complexa" in tipo else MODELO
    return tipo, modelo, perguntar(pergunta, modelo=modelo, max_tokens=120)

for p in ["Qual a capital da França?", "Compare RAG e ajuste fino para um chatbot de suporte interno.",
          "Quanto é 15% de 200?", "Que riscos existem ao usar IA para aprovar compras automaticamente?"]:
    tipo, modelo, _ = rotear(p)
    print(f"{tipo:>9} -> {modelo:<20} | {p}")

### Passo 4. Reduza a latência com caches. · PDF p. 825

#### 🧪 Exercício: Cache exato e cache semântico

Crie um cache que (1) devolve na hora se a **mesma** pergunta já foi feita e (2) reaproveita a resposta se uma pergunta **parecida** (similaridade > 0,9) já foi feita. Meça o tempo das chamadas.

In [ ]:
import time
cache_exato, cache_sem = {}, []   # cache_sem: lista de (vetor, resposta)

def perguntar_com_cache(pergunta):
    if pergunta in cache_exato:
        return cache_exato[pergunta], "exato"
    v = embeddings(pergunta)[0]
    for vetor, resposta in cache_sem:
        if float(similaridade(v, vetor)) > 0.9:
            return resposta, "semântico"
    resposta = perguntar(pergunta, max_tokens=80)
    cache_exato[pergunta] = resposta; cache_sem.append((v, resposta))
    return resposta, "modelo"

for p in ["Qual o horário da loja?", "Qual o horário da loja?", "Qual é o horário de funcionamento da loja?"]:
    t = time.perf_counter(); _, origem = perguntar_com_cache(p)
    print(f"{origem:>10} · {time.perf_counter() - t:.2f}s · {p}")

### Monitoramento e Observabilidade · PDF p. 834

#### 🧪 Exercício: Registrar cada chamada

Escreva `chamar_e_registrar(prompt)` que guarda, para cada chamada: horário, modelo, tokens de entrada/saída e latência. Faça 3 chamadas e mostre o log num DataFrame com o total de tokens.

In [ ]:
import time, datetime
import pandas as pd
LOG = []

def chamar_e_registrar(prompt, modelo=MODELO):
    t = time.perf_counter()
    r = conversar([{"role": "user", "content": prompt}], modelo=modelo, max_tokens=80)
    LOG.append({"quando": datetime.datetime.now().strftime("%H:%M:%S"), "modelo": modelo,
                "tokens_in": r.usage.prompt_tokens, "tokens_out": r.usage.completion_tokens,
                "latencia_s": round(time.perf_counter() - t, 2)})
    return r.choices[0].message.content

for p in ["Defina IA generativa.", "O que é um embedding?", "Para que serve um guarda-corpo?"]:
    chamar_e_registrar(p)
log = pd.DataFrame(LOG); print("Total de tokens:", int(log.tokens_in.sum() + log.tokens_out.sum())); log

### Extraindo feedback conversacional · PDF p. 850

#### 🧪 Exercício: Feedback implícito numa conversa

Dê ao modelo 4 mensagens de usuários (*"não era isso"*, *"perfeito, obrigado"*, *"de novo errado"*, *"pode detalhar mais?"*) e peça para classificar o **sinal de feedback** de cada uma: positivo, negativo ou pedido de mais detalhe (p. 850–857).

In [ ]:
mensagens = ["Não era isso que eu perguntei.", "Perfeito, obrigado!", "De novo errado...", "Pode detalhar mais?"]
sistema = ("Classifique o sinal de feedback da mensagem do usuário sobre a resposta anterior do assistente: "
           "positivo, negativo ou pedido de mais detalhe. Responda só a categoria.")
for m in mensagens:
    print(f"{perguntar(m, sistema=sistema, temperature=0):>24} | {m}")

## 💰 Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou